# Lesson 2 — Files and pandas
### Intro to Computer Science for Non-CS Students

**The bridge from Lesson 1.** We ended last time with two ideas:

1. Every value has a **type**, and the type decides which operations are legitimate.
2. **Storage is not neutral** — saving data converts an in-memory structure into bytes,
   and the file format decides which properties can be reconstructed later.

This whole lesson lives inside those two ideas. A file on disk has *lost* most of its
properties — types, uniqueness, valid ranges, category membership. Our job is to load it,
discover which properties survived, **restore the ones that didn't**, and only then analyze.

```
Data stored in files
        ↓
Load into Python            (Unit 3)
        ↓
Inspect and validate        (Unit 4)
        ↓
Select and transform        (Unit 5)
        ↓
Clean — restore properties  (Unit 6)
        ↓
Summarize and combine       (Units 7–8)
        ↓
Export trustworthy results  (Unit 9)
```

**Prerequisites and setup**

- Lesson 1: types, lists, dictionaries, and the first look at DataFrames.
- `pandas` must be installed (`pip install pandas`).
- Optional, used only in Unit 9: `openpyxl` for Excel and `pyarrow` for Parquet
  (`pip install openpyxl pyarrow`). Those cells skip gracefully if missing.
- Run top to bottom — later cells depend on earlier ones.

**Suggested sessions:** ① Units 1–4 · ② Units 5–7 · ③ Units 8–9.
Each session after the first starts with warm-up questions.

In [ ]:
import json
from pathlib import Path

import pandas as pd

print("pandas", pd.__version__)

# pandas writes FILES but never creates FOLDERS — we make them once, up front.
for folder in ["data/raw", "data/processed", "output"]:
    Path(folder).mkdir(parents=True, exist_ok=True)
print("folders ready:", sorted(str(p) for p in Path("data").iterdir()), "+ output/")

# Optional packages for Unit 9 (the lesson works without them)
for optional in ["openpyxl", "pyarrow"]:
    try:
        __import__(optional)
        print(f"{optional}: available")
    except ImportError:
        print(f"{optional}: not installed — Unit 9 will skip its section")

## Unit 1 — Files and persistence  ·  ~20 min

Everything we built in Lesson 1 — lists, dictionaries, DataFrames — lives in **memory (RAM)**.
Memory is fast but temporary: restart the kernel and it is gone.

A **file** is data written to disk so it can be reused later, shared with another program
or person, audited, and kept as a historical record.

In [ ]:
# Lesson 1 ended with this shape: a dataset as a list of dictionaries.
orders = [
    {"id": "T1", "amount": 100},
    {"id": "T2", "amount": 150},
]
print(orders)
# This exists only in memory. Restart the kernel and `orders` no longer exists.

In [ ]:
# Writing bytes to disk makes data persistent.
note_path = Path("data/raw/first_file.txt")
note_path.write_text("Files are just bytes plus an agreed encoding.\n")

print("content :", note_path.read_text().strip())
print("size    :", note_path.stat().st_size, "bytes")
print("exists  :", note_path.exists())

🧭 Two things that are easy to confuse:

```
File format:    HOW information is encoded as bytes on disk
Data structure: HOW information is organized inside a running program
```

pandas is the bridge between the two. The same customer table can exist as a CSV file,
a JSON file, an Excel workbook, a Parquet file, or a DataFrame — the *information* is
similar, the *representation* changes. That is Lesson 1's "same data, different
structures" idea, now applied to disk.

## Unit 2 — Four file formats  ·  ~30 min

| Format  | Best for                 | Strength                       | Limitation                  |
| ------- | ------------------------ | ------------------------------ | --------------------------- |
| CSV     | Simple tables            | Universal and human-readable   | Weak type preservation      |
| JSON    | APIs and nested records  | Flexible nesting               | Awkward when deeply nested  |
| Excel   | Human business workflows | Sheets, formulas, presentation | Easy to make inconsistent   |
| Parquet | Large analytical tables  | Typed, compressed, efficient   | Not human-readable          |

We will read CSV and JSON now, and meet Excel and Parquet in Unit 9 when we export.

### Our dataset for the whole lesson

A small online stationery shop. Two tables:

- **customers** — who they are and where
- **transactions** — what they bought

The files below are **deliberately messy**, because real files arrive messy. We write them
to disk ourselves so you can see exactly what the raw bytes look like before pandas
touches them.

In [ ]:
customers_csv = '''customer_id,name,region,signup_date
00101,Alice,West,2025-11-03
00102,Bob,East,2026-01-15
00103,Carol, west,2026-02-20
00104,David,WEST,2026-03-08
00105,Elena,east ,2026-04-12
00106,Frank,North,2026-05-01
'''

Path("data/raw/customers.csv").write_text(customers_csv)
print(customers_csv)

In [ ]:
transactions_csv = '''transaction_id,customer_id,date,quantity,unit_price,discount
T001,00101,2026-05-02,2,50.0,0.10
T002,00102,2026-05-05,1,80.0,
T003,00101,2026-05-11,3,40.0,-
T004,00103,2026-05-18,5,20.0,0.05
T005,00104,2026-06-01,2,not-available,
T006,00999,2026-06-03,1,25.0,0.15
T007,00105,2026-06-09,-2,50.0,
T007,00105,2026-06-09,-2,50.0,
T008,00106,2026-06-15,4,30.0,0.20
T009,00102,2026-06-21,2,45.0,-
T010,00103,invalid,1,60.0,0.10
'''

Path("data/raw/transactions.csv").write_text(transactions_csv)
print(transactions_csv)

🔮 **Predict before moving on.** Read the two raw files above like a detective and write
down every problem you can spot. Hints: look at the IDs, the dates, the numbers, the
empty spots, the repeated lines, and the region spellings. Keep your list — Unit 4 will
find them *systematically*, and you can compare.

In [ ]:
# The SAME customer table, encoded as JSON instead of CSV.
customer_records = [
    {"customer_id": "00101", "name": "Alice", "region": "West",  "signup_date": "2025-11-03"},
    {"customer_id": "00102", "name": "Bob",   "region": "East",  "signup_date": "2026-01-15"},
    {"customer_id": "00103", "name": "Carol", "region": " west", "signup_date": "2026-02-20"},
    {"customer_id": "00104", "name": "David", "region": "WEST",  "signup_date": "2026-03-08"},
    {"customer_id": "00105", "name": "Elena", "region": "east ", "signup_date": "2026-04-12"},
    {"customer_id": "00106", "name": "Frank", "region": "North", "signup_date": "2026-05-01"},
]

json_text = json.dumps(customer_records, indent=2)
Path("data/raw/customers.json").write_text(json_text)
print(json_text[:220], "...")

🧭 One table, two encodings. CSV is a rectangle of text; JSON is the nested
dictionary-and-list tree from Lesson 1, written to disk. Neither file stores *types*:
in both, `00101` is just characters, and nothing on disk says "this column must be
unique" or "region has exactly four legal values". Those **properties live in our heads**
until we restore them in Python.

## Unit 3 — Paths and loading  ·  ~35 min

A **path** tells the program where a file lives.

```
data/raw/customers.csv                          ← relative (starts from the working directory)
/Users/you/project/data/raw/customers.csv       ← absolute (starts from the disk root)
```

`pathlib` treats paths as objects instead of raw strings.

In [ ]:
from pathlib import Path

RAW = Path("data/raw")

file_path = RAW / "customers.csv"       # the / operator joins path pieces
print("working directory:", Path.cwd())
print("relative path    :", file_path)
print("exists?          :", file_path.exists())
print("name / suffix    :", file_path.name, "/", file_path.suffix)
print("parent folder    :", file_path.parent)
print("raw files        :", sorted(p.name for p in RAW.iterdir()))

In [ ]:
# The most common loading error — read it from the BOTTOM line up.
try:
    pd.read_csv(RAW / "customerz.csv")     # typo on purpose
except FileNotFoundError as error:
    print(type(error).__name__, "->", error)
    print()
    print("Diagnosis: the path is wrong or the file is not where you think it is.")
    print("First aid:  print(Path.cwd()) and check .exists() on the path.")

In [ ]:
# Naive load: let pandas guess everything.
customers_naive = pd.read_csv(RAW / "customers.csv")
print(customers_naive)
print()
print(customers_naive.dtypes)

🧭 Look at `customer_id`: we wrote `00101` and got the *number* `101` — dtype `int64`.
This is exactly the ZIP-code trap from the end of Lesson 1, happening to us on our own data.

`dtypes` is Lesson 1's central idea at column scale: **the type is attached to the value**
— pandas attaches one type tag to each *column*. CSV stores no types, so loading is a
**reconstruction**, and pandas guessed "looks numeric" for our IDs. An ID is a label, not
a quantity — you never add customer IDs — so the guess is wrong. Important assumptions
should be stated, not guessed.

In [ ]:
# Naive load of transactions: more silent guesses.
tx_naive = pd.read_csv(RAW / "transactions.csv")
print(tx_naive.dtypes)
print()
print(tx_naive[["unit_price", "discount"]].head(4))

🧭 `unit_price` and `discount` came back as `object` (pandas-speak for "text or mixed").
**One junk value poisons the whole column**: because `not-available` cannot be a number,
pandas keeps *every* price as text — a column, like a NumPy array in Lesson 1, wants all
its values to share one type. And `-` in `discount` is this file's way of writing
"missing", but pandas only auto-recognizes its default missing tokens (empty string,
`NA`, `NaN`, ...). Unknown tokens must be declared.

In [ ]:
# Explicit load: state the assumptions instead of letting pandas guess.
customers = pd.read_csv(
    RAW / "customers.csv",
    dtype={"customer_id": "string"},     # IDs are labels, not quantities
    parse_dates=["signup_date"],         # clean column -> parse at load time
)

tx = pd.read_csv(
    RAW / "transactions.csv",
    dtype={"transaction_id": "string", "customer_id": "string"},
    na_values=["-"],                     # declare THIS file's missing-value token
    # note: we deliberately do NOT parse `date` here — it contains junk
    # ("invalid") and we want to handle that visibly in Unit 6.
)

print(customers.dtypes)
print()
print(tx.dtypes)

In [ ]:
# JSON: flat, record-shaped JSON loads directly.
customers_from_json = pd.read_json(
    RAW / "customers.json",
    dtype={"customer_id": "string"},     # the JSON reader would also guess 00101 -> 101!
)
print(customers_from_json.head(3))
print()
print(customers_from_json.dtypes)

In [ ]:
# Nested JSON (one customer CONTAINING their orders) is a tree, not a rectangle.
nested = {
    "customer_id": "00101",
    "name": "Alice",
    "orders": [
        {"id": "T001", "amount": 90.0},
        {"id": "T003", "amount": 120.0},
    ],
}

# json_normalize flattens the tree into a table: one row per order,
# with the parent fields repeated alongside.
flat = pd.json_normalize(nested, record_path="orders", meta=["customer_id", "name"])
print(flat)

🧭 `read_json` fits **flat** record lists. Deeply nested JSON must be *flattened* first —
`json_normalize` walks the dictionary-and-list tree from Lesson 1 and lays it out as a
rectangle.

**Loading is a series of decisions**, whether you make them or pandas guesses them:
where the header is · what the delimiter is · which tokens mean "missing" · what type
each column has · how dates are parsed · what text encoding is used. The explicit
version of a load is documentation of your assumptions.

## Unit 4 — Interview the DataFrame before trusting it  ·  ~40 min

The most common analytical mistake: calculating immediately after loading.

```
Load  →  Inspect  →  Validate  →  only then Analyze
```

Each inspection command answers one question:

| Command          | Question it answers                          |
| ---------------- | -------------------------------------------- |
| `head()` `tail()`| What do actual rows look like?                |
| `shape`          | How many rows and columns?                    |
| `columns`        | Which variables exist?                        |
| `dtypes`         | How did pandas interpret each column?         |
| `info()`         | Types + non-null counts in one report         |
| `describe()`     | Are the numeric ranges plausible?             |
| `value_counts()` | Which category labels exist, how often?       |

In [ ]:
print(tx.head(3))
print()
print(tx.tail(2))
print()
print("shape  :", tx.shape)          # (rows, columns)
print("columns:", list(tx.columns))

In [ ]:
tx.info()

🔮 **Predict before running:** for a shop, what should the *minimum* of `quantity` be?
And how many distinct `region` labels should the customer table have?

In [ ]:
# describe() covers NUMERIC columns only by default...
print(tx.describe())

# ...text-like columns need to be asked for explicitly.
print()
print(tx.describe(include=["object", "string"]))

In [ ]:
# value_counts: the x-ray for categorical columns.
print(customers["region"].value_counts())
print()
print("distinct region labels:", customers["region"].nunique())

🧭 Six "different" regions — but ` west`, `WEST`, and `West` are the same place wearing
different costumes. In Lesson 1 terms, `region` should behave like a **set membership**
property: every value drawn from a small set of legal labels. The file lost that property;
`value_counts()` is how we notice. (And `describe()` just showed `quantity` has a minimum
of **−2** — remember that for Unit 6.)

In [ ]:
# Missing values and duplicates — counts first, decisions later.
print("missing per column:")
print(tx.isna().sum())
print()
print("fully duplicated rows :", tx.duplicated().sum())
print("duplicated transaction_id:", tx["transaction_id"].duplicated().sum())

### The seven questions to ask any new dataset

1. Is the **row count** plausible?
2. Are the **column names** what you expect?
3. Are the **types** correct (`dtypes`)?
4. Are important values **missing** (`isna().sum()`)?
5. Are identifiers that should be unique actually **unique**?
6. Do **numeric ranges** make sense (`describe()`)?
7. Are **dates and categories** within their expected sets?

Compare with your detective list from Unit 2 — the inspection commands should have
rediscovered every planted problem, without you needing to eyeball the raw file.

---
## ⏱️ Session 2 warm-up  ·  5 questions from last time

Answer from memory first, then check.

1. We wrote `00101` into the file and got `101` back. Why, and what is the fix?
2. What does it mean when a column's dtype is `object`?
3. Which single command reveals that ` west`, `WEST`, and `West` all exist in a column?
4. CSV and JSON both stored our customer table. What shape does each format naturally hold?
5. Why do we inspect before we analyze — what exactly can go wrong if we skip it?

<details><summary><b>Answers</b></summary>

1. CSV stores no types; pandas guessed "numeric" and dropped the leading zeros.
   Fix: declare `dtype={"customer_id": "string"}` at load time — IDs are labels, not quantities.
2. Text or mixed values — often a sign that junk (like `not-available`) poisoned a numeric column.
3. `column.value_counts()`.
4. CSV: a flat rectangle of rows × columns. JSON: a nested tree of dictionaries and lists.
5. Silent wrong guesses (types, missing tokens), duplicates, and impossible values flow
   straight into your results without any error message.
</details>

## Unit 5 — Selecting, filtering, sorting, transforming  ·  ~45 min

### 5.1 Selecting columns

In [ ]:
one_column = tx["quantity"]              # single brackets -> a Series (1-D, one dtype)
two_columns = tx[["transaction_id", "quantity"]]   # list of names -> a smaller DataFrame

print(type(one_column).__name__, "| dtype:", one_column.dtype)
print(type(two_columns).__name__, "| shape:", two_columns.shape)

🧭 A Series is one labeled column with a single dtype — the DataFrame is a dictionary-like
collection of them. Select only the variables the current question needs.

### 5.2 Selecting rows: position vs label

- `iloc` = **integer position** (like list indexing in Lesson 1)
- `loc`  = **labels** (like dictionary keys in Lesson 1)

They look interchangeable on a fresh DataFrame — filtering exposes the difference.

In [ ]:
print(tx.iloc[0])        # first row by POSITION
print()
print(tx.iloc[0:3][["transaction_id", "quantity"]])   # first three rows

In [ ]:
# Filter: keep rows where a condition is True.
bulk = tx[tx["quantity"] >= 3]
print(bulk[["transaction_id", "quantity"]])
print()
print("index labels that survived the filter:", list(bulk.index))

In [ ]:
# The labels kept their old values — positions renumber, labels do not.
print("bulk.iloc[0] -> row at position 0:")
print(bulk.iloc[0][["transaction_id", "quantity"]])
print()

try:
    bulk.loc[0]                      # there is NO label 0 anymore
except KeyError:
    print("bulk.loc[0] -> KeyError: no row carries the LABEL 0")

# If you want tidy 0..n-1 labels after filtering:
bulk_reset = bulk.reset_index(drop=True)
print()
print("after reset_index:", list(bulk_reset.index))

🧭 `iloc` counts positions; `loc` looks up labels. After filtering, positions are
renumbered but labels travel with their rows — exactly the list-vs-dictionary access
distinction from Lesson 1.

### 5.3 Combining conditions

In [ ]:
# & = and, | = or, ~ = not. Parentheses around EACH condition are required.
may_bulk = tx[(tx["quantity"] >= 3) & (tx["date"] < "2026-06-01")]
print(may_bulk[["transaction_id", "date", "quantity"]])

In [ ]:
# A filter that "should" find three West customers...
west = customers[customers["region"] == "West"]
print(west[["name", "region"]])
print()
print("rows found:", len(west), " (value_counts told us to expect 3!)")

🧭 Only Alice matched — ` west` and `WEST` fail the `== "West"` test. **Messy text
silently breaks filters**: no error, just missing rows. This is why Unit 6 exists.

### 5.4 Sorting

In [ ]:
print(tx.sort_values(by="quantity", ascending=False).head(3)[["transaction_id", "quantity"]])
print()
# Multiple keys: by customer, largest orders first within each customer.
print(tx.sort_values(by=["customer_id", "quantity"], ascending=[True, False])
        .head(5)[["customer_id", "transaction_id", "quantity"]])

### 5.5 Creating new columns — vectorized operations

One expression applies to the whole column at once (the NumPy idea from Lesson 1).

In [ ]:
# Works: discount is float64, so arithmetic is legitimate.
print((tx["discount"] * 100).head(4))    # as percentages

🔮 **Predict before running:** `unit_price` is still `object` (text). What will
`quantity * unit_price` produce — an error, or something else?

In [ ]:
revenue_wrong = tx["quantity"] * tx["unit_price"]
print(revenue_wrong)

🧭 **No error — worse.** In Python, `2 * "50.0"` is *legitimate*: string repetition,
giving `"50.050.0"`. So pandas happily "multiplied" every row, and a negative quantity
times a string produced an empty string. Lesson 1's first checklist question — *which
operations are legitimate for this type?* — is not philosophy; the wrong dtype produces
**silently wrong answers**, which are far more dangerous than errors. We fix the type in
Unit 6 and compute revenue properly there.

### 5.6 Conditional columns — and the one assignment rule

In [ ]:
# Correct pattern: assign THROUGH df.loc[row_condition, column].
tx["order_size"] = "standard"
tx.loc[tx["quantity"] >= 3, "order_size"] = "bulk"
print(tx["order_size"].value_counts())

In [ ]:
# The classic trap: filtering FIRST and assigning SECOND ("chained" assignment).
tx_demo = tx.copy()
tx_demo[tx_demo["quantity"] >= 3]["order_size"] = "XXL"     # looks plausible, is WRONG

print("rows actually changed to XXL:", (tx_demo["order_size"] == "XXL").sum())

🧭 Zero rows changed (pandas may also print a `SettingWithCopyWarning`). The filter
built a *temporary copy*, and the assignment modified the copy, which was then thrown away.

> **The rule:** to modify selected rows, always write
> `df.loc[condition, "column"] = value` — never `df[condition]["column"] = value`.

## Unit 6 — Cleaning: restoring the properties the file lost  ·  ~55 min

Everything Unit 4 found is a property violation:

| Finding                      | Property to restore                    |
| ---------------------------- | -------------------------------------- |
| prices stored as text        | column has one numeric type            |
| `invalid` in the date column | dates are real points in time          |
| `-` and blanks               | missingness is explicit, not disguised |
| repeated `T007` line         | transaction IDs are unique             |
| ` west` / `WEST` / `West`    | categories from a fixed legal set      |
| quantity −2                  | values within a meaningful range       |

### 6.1 Fixing types — with the count-before-and-after habit

In [ ]:
# to_numeric with errors="coerce": junk becomes NaN (visible), instead of crashing.
missing_before = tx["unit_price"].isna().sum()
tx["unit_price"] = pd.to_numeric(tx["unit_price"], errors="coerce")
missing_after = tx["unit_price"].isna().sum()

print(f"unit_price missing: {missing_before} -> {missing_after}  (dtype: {tx['unit_price'].dtype})")
print()
print("the rows coerce flagged:")
print(tx.loc[tx["unit_price"].isna(), ["transaction_id", "customer_id"]])

In [ ]:
# Same pattern for dates: junk becomes NaT ("not a time").
missing_before = tx["date"].isna().sum()
tx["date"] = pd.to_datetime(tx["date"], errors="coerce")
missing_after = tx["date"].isna().sum()

print(f"date missing: {missing_before} -> {missing_after}  (dtype: {tx['date'].dtype})")
print()
print(tx.loc[tx["date"].isna(), ["transaction_id", "customer_id"]])

🧭 `errors="coerce"` converts damage into *visible* missing values. The habit that keeps
it honest: **count missing before and after, and look at the flagged rows.** Coercing and
never looking is how problems get silently swept under the rug.

### 6.2 Missing values — a decision, not a reflex

Ask four questions before touching a missing value:

1. *Why* is it missing?
2. Does missing mean **zero**, **unknown**, or **not applicable**?
3. Would dropping the row bias the analysis?
4. Can the true value be recovered from the source?

In [ ]:
print(tx.isna().sum())
print()

# discount: in this shop, an empty discount field means "no discount was applied".
# That is a BUSINESS fact justifying zero — not a pandas default.
tx["discount"] = tx["discount"].fillna(0)

# unit_price: unknown is NOT zero. A free product would corrupt every revenue number,
# so we leave it missing and let it stay visible downstream.
# date NaT: also stays — an invented date would be worse than an honest gap.

print("after the discount decision:")
print(tx.isna().sum())

### 6.3 Duplicates — uniqueness is a property tied to meaning

Two customers may share a name; one customer may have many transactions. But one
transaction ID should mean one event. Duplicate checks come from what a row *means*
(the set idea from Lesson 1), not from a reflex.

In [ ]:
print("exact duplicate rows:")
print(tx[tx.duplicated(keep=False)][["transaction_id", "customer_id", "quantity"]])

rows_before = len(tx)
tx = tx.drop_duplicates()
print()
print(f"rows: {rows_before} -> {len(tx)}")
print("transaction_id still duplicated:", tx["transaction_id"].duplicated().sum())

### 6.4 Text normalization — one costume per category

In [ ]:
before = customers["region"].value_counts()

customers["region"] = customers["region"].str.strip().str.title()

after = customers["region"].value_counts()
print("BEFORE:", dict(before))
print("AFTER :", dict(after))

In [ ]:
# Rule validation: region must come from the approved set (set membership, Lesson 1).
valid_regions = {"East", "West", "North", "South"}
outside = customers[~customers["region"].isin(valid_regions)]
print("regions outside the approved set:", len(outside))

# Range validation: are the dates inside the expected window?
print("date range:", tx["date"].min(), "->", tx["date"].max())

### 6.5 Range checks — and when an anomaly is not an error

In [ ]:
suspicious = tx[tx["quantity"] < 0]
print(suspicious[["transaction_id", "customer_id", "quantity", "unit_price"]])

🧭 We *asked the shop* about T007: it is a **refund** — Elena returned two notebooks.
Negative quantity is legitimate here and must stay, or we would overstate revenue.

> Cleaning is not "delete everything strange". It is: notice → investigate → decide →
> **record the decision**. (This notebook *is* the record: anyone can rerun it and see
> exactly what was changed and why.)

### 6.6 Now — and only now — compute revenue

In [ ]:
tx["revenue"] = tx["quantity"] * tx["unit_price"] * (1 - tx["discount"])

print(tx[["transaction_id", "quantity", "unit_price", "discount", "revenue"]])

🧭 Compare with the garbage from Unit 5: same expression, correct dtypes, correct result.
T005's revenue is `NaN` because its price is unknown — the honest answer propagates
instead of a fake zero. And T007's revenue is **−100**: the refund correctly *subtracts*.

### The reusable quality-check block

In [ ]:
# Run this block on ANY dataset after cleaning. Ten seconds, catches most disasters.
print("shape:", tx.shape)
print()
print(tx.dtypes)
print()
print("missing:")
print(tx.isna().sum())
print()
print("duplicate rows:", tx.duplicated().sum(),
      "| duplicate IDs:", tx["transaction_id"].duplicated().sum())
print()
print(tx.describe())

## Unit 7 — Grouping and summarizing  ·  ~35 min

### Whole-column aggregation

In [ ]:
print("total revenue :", tx["revenue"].sum())      # note: sum() skips NaN silently
print("mean revenue  :", round(tx["revenue"].mean(), 2))
print("known revenues:", tx["revenue"].count(), "of", len(tx), "rows")

### groupby — split, apply, combine

```
Split rows into groups        (one group per customer)
        ↓
Apply a calculation per group (sum the revenue)
        ↓
Combine results into a table  (one row per customer)
```

In [ ]:
revenue_per_customer = tx.groupby("customer_id")["revenue"].sum()
print(revenue_per_customer)

In [ ]:
# Dates unlock time grouping: .dt accesses date parts, to_period("M") = calendar month.
tx["month"] = tx["date"].dt.to_period("M")

# dropna=False keeps the NaT group visible — T010 has no valid date,
# and silently losing its revenue would make the monthly totals lie.
print(tx.groupby("month", dropna=False)["revenue"].sum())

In [ ]:
# Several summaries at once, with named results.
customer_summary = (
    tx.groupby("customer_id")
    .agg(
        total_revenue=("revenue", "sum"),
        n_transactions=("transaction_id", "nunique"),
        avg_discount=("discount", "mean"),
    )
    .reset_index()
)
print(customer_summary)

### Pivot tables — a groupby on two keys, shown as a grid

In [ ]:
grid = pd.pivot_table(
    tx,
    values="revenue",
    index="customer_id",   # one key down the side...
    columns="month",       # ...one key across the top
    aggfunc="sum",
)
print(grid)

🧭 A pivot table is nothing new: group by *two* keys, display as a grid. Note that the
NaT row (T010) is dropped from the pivot by default — one more place where an invalid
date quietly exits the analysis. Use pivots once `groupby` feels natural, not before.

---
## ⏱️ Session 3 warm-up  ·  5 questions from last time

1. `loc` vs `iloc` — which uses labels, which uses positions, and when do they disagree?
2. Why count missing values before *and* after `errors="coerce"`?
3. We filled missing `discount` with 0 but left missing `unit_price` alone. Why?
4. Describe `groupby` in three words.
5. What is the one correct pattern for assigning to a filtered set of rows?

<details><summary><b>Answers</b></summary>

1. `loc` = labels, `iloc` = integer positions. They disagree after filtering or sorting,
   because labels travel with rows while positions renumber.
2. The difference = how many values coerce turned into NaN/NaT. No count → silent data loss.
3. Missing discount means "no discount" (a business fact → 0). Missing price means
   "unknown" — zero would fabricate a free product and corrupt revenue.
4. Split, apply, combine.
5. `df.loc[condition, "column"] = value`.
</details>

## Unit 8 — Combining tables with merge  ·  ~40 min

Customers and transactions live in separate files, linked by `customer_id` — a key,
exactly like a dictionary key from Lesson 1: it points to the matching record.

```
transactions  +  customers   →   transactions WITH customer info
        (matched on customer_id)
```

In [ ]:
# Inner join: keep only rows that match on BOTH sides.
inner = tx.merge(customers, on="customer_id", how="inner")

print("transactions:", len(tx), "rows  ->  inner join:", len(inner), "rows")
lost = set(tx["transaction_id"]) - set(inner["transaction_id"])
print("who disappeared:", lost)

In [ ]:
# Left join: keep EVERY transaction; attach customer info where it exists.
# indicator=True adds a _merge audit column saying where each row found its match.
sales = tx.merge(customers, on="customer_id", how="left", indicator=True)

print("rows:", len(sales), " (must still be", len(tx), "— a left join keeps every left row)")
print()
print(sales["_merge"].value_counts())
print()
print("the unmatched row:")
print(sales.loc[sales["_merge"] == "left_only",
                ["transaction_id", "customer_id", "name", "region"]])

🧭 T006 belongs to customer `00999`, who is not in the customer table (a typo? a deleted
account?). The **inner** join silently *dropped* that sale; the **left** join kept it with
`NaN` customer info, and `indicator` flagged it. A join does not just add columns —
**it can remove rows, and it can multiply them:**

In [ ]:
# The multiplication trap, on tiny toy tables: the key is duplicated on BOTH sides.
left = pd.DataFrame({"key": ["A", "A"], "left_val": [1, 2]})
right = pd.DataFrame({"key": ["A", "A"], "right_val": [10, 20]})

blown_up = left.merge(right, on="key")
print(f"2 rows x 2 rows -> {len(blown_up)} rows:")
print(blown_up)

In [ ]:
# Protection: DECLARE the expected relationship. many_to_one means:
# many transactions may share a customer, but each customer_id appears ONCE in customers.
# If that expectation is ever violated, merge raises an error instead of inflating revenue.
sales = tx.merge(customers, on="customer_id", how="left",
                 validate="many_to_one", indicator=True)
print("validated merge OK:", len(sales), "rows")

In [ ]:
# Regional summary — then RECONCILE against the pre-join total.
region_summary = (
    sales.groupby("region")
    .agg(
        total_revenue=("revenue", "sum"),
        n_transactions=("transaction_id", "nunique"),
        n_customers=("customer_id", "nunique"),
    )
    .reset_index()
)
print(region_summary)
print()
print("sum of regional revenue:", region_summary["total_revenue"].sum())
print("total revenue before   :", tx["revenue"].sum())

🔮 **21.25 is missing.** Where did it go?

<details><summary><b>Answer</b></summary>

T006's region is `NaN` (unmatched customer), and `groupby` drops NaN groups by default —
so its 21.25 of revenue silently fell out of the regional table. The totals *look*
authoritative and are quietly incomplete.
</details>

In [ ]:
# dropna=False makes the missing group visible — now the totals reconcile.
honest_summary = (
    sales.groupby("region", dropna=False)["revenue"].sum().reset_index()
)
print(honest_summary)
print()
print("reconciles now:",
      honest_summary["revenue"].sum() == tx["revenue"].sum())

🧭 **After every join + group, reconcile the grand total.** It costs one line and catches
both silent row loss (like this) and silent row multiplication (the toy example above).

## Unit 9 — Exporting and reproducibility  ·  ~25 min

The result of all this work should be (a) clean output files, and (b) a **repeatable
recipe** — this notebook — that regenerates them from the untouched raw files.

In [ ]:
# The _merge audit column did its job; drop bookkeeping before export.
# month is derivable from date, so we export the source of truth, not the derivation.
sales_export = sales.drop(columns=["_merge", "month"])

out_path = Path("data/processed/sales_clean.csv")
sales_export.to_csv(out_path, index=False)     # index=False: row labels are not data
print("wrote", out_path, "-", out_path.stat().st_size, "bytes")

🔮 **Predict:** we clean-loaded, fixed every type, and saved to CSV. If we now reload
that CSV *naively*, what happens to `customer_id` and `date`?

In [ ]:
reloaded = pd.read_csv(out_path)

comparison = pd.DataFrame({
    "before_saving": sales_export.dtypes,
    "after_reload": reloaded.dtypes,
})
print(comparison)
print()
print("customer_id round trip:", sales_export["customer_id"].iloc[0],
      "->", reloaded["customer_id"].iloc[0])

🧭 **CSV forgot everything again** — IDs are numbers, dates are text. That is not a bug;
it is the format. Lesson 1 said it: *storage is not neutral — the format decides which
properties can be reconstructed.* CSV preserves characters, nothing more. Which is
exactly what Parquet is for:

In [ ]:
try:
    import pyarrow  # noqa: F401

    pq_path = Path("data/processed/sales_clean.parquet")
    sales_export.to_parquet(pq_path, index=False)
    from_parquet = pd.read_parquet(pq_path)

    print(pd.DataFrame({
        "before_saving": sales_export.dtypes,
        "after_reload": from_parquet.dtypes,
    }))
    print()
    print("customer_id round trip:", sales_export["customer_id"].iloc[0],
          "->", from_parquet["customer_id"].iloc[0])
except ImportError:
    print("pyarrow not installed -> skipping. Install with: pip install pyarrow")

In [ ]:
# Excel: for handing results to humans.
try:
    import openpyxl  # noqa: F401

    xlsx_path = Path("output/region_summary.xlsx")
    region_summary.to_excel(xlsx_path, index=False)
    print("wrote", xlsx_path)
except ImportError:
    print("openpyxl not installed -> skipping. Install with: pip install openpyxl")

### The reproducibility contract

```
project/
├── data/
│   ├── raw/          ← original files: READ-ONLY, never edited, never overwritten
│   └── processed/    ← cleaned tables, regenerable by rerunning this notebook
├── output/           ← reports and summaries for humans
└── notebook          ← THE RECIPE: every assumption and decision, executable
```

- The **raw files are the evidence** — cleaning happens in code, never in the file.
- Every output must be regenerable: delete `processed/` and `output/`, rerun this
  notebook, get identical files. A hand-edited spreadsheet can never make that promise.
- The decisions we made — discount NaN means 0, T007 is a legitimate refund, T005's
  price stays unknown — live here as running code, not in anyone's memory.

## Wrap-up

### Six misconceptions this lesson should have broken

| Misconception | Reality |
| --- | --- |
| "If pandas loads it, it's correct" | pandas decodes bytes; it cannot know what the columns *mean* |
| "Missing values should be filled" | discount-NaN meant 0; price-NaN meant unknown — meaning decides |
| "Duplicates should be removed" | the repeated `T007` *line* was an error; the refund itself was real |
| "A join just adds columns" | inner dropped T006; duplicate keys multiplied 2×2→4 |
| "The index is an ID" | the index is a row label; business IDs stay as explicit columns |
| "CSV preserves my DataFrame" | CSV forgot our types twice in one lesson; Parquet kept them |

### The workflow, one sentence

> *Identify the format, load with explicit types, inspect and validate, clean without
> touching the raw file, combine and reconcile, export a reproducible result.*

That sentence — not any individual method — is the skill. Every new dataset gets the
same interview: the seven questions of Unit 4, then the quality block of Unit 6.

### Where each Lesson 1 property went

- *type attached to the value* → `dtypes`, explicit loading, `to_numeric` / `to_datetime`
- *legitimate operations* → `2 * "50.0"` and the garbage-revenue column
- *uniqueness (sets)* → duplicate transaction IDs, `value_counts`, `isin` validation
- *keys → records (dictionaries)* → `customer_id` as the merge key
- *storage is not neutral* → CSV amnesia vs the Parquet round trip

### Resources

- [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) — best first overview
- [pandas I/O guide](https://pandas.pydata.org/docs/user_guide/io.html) — every reader and writer
- [Missing-data guide](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [GroupBy guide](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Merge guide](https://pandas.pydata.org/docs/user_guide/merging.html)
- [Python pathlib](https://docs.python.org/3/library/pathlib.html)

**Next:** the practice notebook — new messy files, same seven questions, no answers
pre-filled.